## Comprehensive analysis

- Input: any of $z_{enc}$, $z_{pred}$, $z_{target}$, or $z_{pred} - z_{target}$ + external metadata (that can be iterated over)
- Tools: LASSO on PCA axes, HDBSCAN cluster enrichment, SAE feature correlation

*"how much of the learned geometry aligns with known labels, and what's left over?"*

In [ ]:
import numpy as np
from pathlib import Path

from src.analysis.eval_infra import (
    load_label, 
    load_escalation_labels, 
    compute_escalation_criterions)
from src.utils.io import (EXPERIMENTS_DIR, DATA_DIR, 
                          load_embeddings, load_sequences_dict, 
                          load_metadata, load_json)
from src.utils.seed import load_exp_seed, set_global_seed

In [ ]:
# -- Config Settings --
MODEL       = "test_01"
EMB_NAME    = "embeddings_40.npz"

In [ ]:
EXPERIMENTS     = Path("experiments")
MODEL_DIR       = EXPERIMENTS / MODEL
ANALYSIS_DIR    = MODEL_DIR / "analysis" / "comp"
FIGURES_DIR     = ANALYSIS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCES_PATH  = Path(DATA_DIR / "sequences.jsonl")

set_global_seed(load_exp_seed(EXPERIMENTS_DIR / f'{MODEL}.yaml'))

# -- Load embeddings --
emb, EMB_PATH = load_embeddings(MODEL_DIR, EMB_NAME)
print(f"Embeddings: {EMB_PATH.name}")

z_encs = emb["z_encs"]             # (N, C_padded, D)
z_pred = emb["z_pred"]             # (N, D)
z_target = emb["z_target"]         # (N, D)
ctx_pad_mask = emb["ctx_pad_mask"]  # (N, C_padded)
subject_ids = emb["subject_ids"]   # (N,)
mask_pos = emb["mask_pos"]         # (N,)
pred_error = z_pred - z_target     # (N, D)

# Flatten valid encounters from z_encs
valid = ~ctx_pad_mask.astype(bool)
z_enc_flat = z_encs[valid]         # (N_valid, D)
enc_subject_ids = np.broadcast_to(
    subject_ids[:, None], ctx_pad_mask.shape)[valid]
enc_positions = np.broadcast_to(
    np.arange(ctx_pad_mask.shape[1])[None, :], ctx_pad_mask.shape)[valid]

print(f"  z_encs:      {z_encs.shape}")
print(f"  z_enc_flat:  {z_enc_flat.shape}")
print(f"  z_pred:      {z_pred.shape}")
print(f"  z_target:    {z_target.shape}")
print(f"  pred_error:  {pred_error.shape}")
print(f"  subjects:    {len(np.unique(subject_ids))}")

# -- Load labels from sequences.jsonl
patients = load_sequences_dict(SEQUENCES_PATH)

unique_sids = np.unique(subject_ids)
label_escalation_patient = np.array([
    patients[sid].get("label_escalation", 0) for sid in unique_sids])
label_30d_patient = np.array([
    patients[sid].get("label_30d", 0) for sid in unique_sids])
 
# Sample-level encounter labels (for the masked encounter)
label_esc_per_sample = np.array([
    patients[str(sid)]["label_escalation_per_enc"][int(mp)]
    if "label_escalation_per_enc" in patients[str(sid)] else 0
    for sid, mp in zip(subject_ids, mask_pos)])
 
print(f"  Escalation rate (patient): {label_escalation_patient.mean():.3f}")
print(f"  Escalation rate (encounter): {label_esc_per_sample.mean():.3f}")
print(f"  30d readmit rate: {label_30d_patient.mean():.3f}")

In [ ]:
try:
    meta, meta_names, meta_pids = load_metadata(DATA_DIR)
    print(f"  Metadata: {meta.shape[0]} patients x {meta.shape[1]} features")
except FileNotFoundError:
    print("  Metadata not found")

In [ ]:
results = load_json(ANALYSIS_DIR / "representation.json")
if results is None:
    raise FileNotFoundError(f"No representation results found")
seed_stability = load_json(
    ANALYSIS_DIR / "seed_stability.json")
s1_results = load_json(
    MODEL_DIR / "analysis" / "representation" / "representation.json")
s3_results = load_json(
    MODEL_DIR / "analysis" / "error_decomp" / "error.json")

## Plotting

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

**Information flow heatmap**
- rows = vectors $\rightarrow$ $z_{enc}$ (pooled), $z_{pred}$ (pooled), $z_{target}$ (pooled), $P-T$ (pooled)
- columns = labels (escalation, 30d readmission)
- Cell color = AUROC

Annotate each cell with AUROC $\pm$ std. This is the paper's main results table rendered as a figure.

In [ ]:
from src.analysis.plotting import _s4_info_flow_heatmap
_s4_info_flow_heatmap(results,
                      show=True, save=False, fig_dir=FIGURES_DIR)

**Mislabeling gap progression**: three grouped bars (one per tier: LASSO $R^2$, unlabeled cluster fraction, unlabeled SAE feature fraction), with groups for $z_{enc}$ vs pred_error.

Shows whether error geometry is more or less clinically labeled than encoder geometry.

In [ ]:
from src.analysis.plotting import _s4_mislabeling_gap
_s4_mislabeling_gap(results, s1_results, s3_results,
                    show=True, save=False, fig_dir=FIGURES_DIR)

**ICD block reconstruction**: bar chart of per-chapter AUROC for $z_{pred}$ vs $z_{target}$.

Shows which ICD chapters the predictor can/can't anticipate.

In [ ]:
from src.analysis.plotting import _s4_icd_block
_s4_icd_block(results,
              show=True, save=False, fig_dir=FIGURES_DIR)

**Escalation type decomposition**: bar chart of per-criterion AUROC on $z_{pred}$. Shows which escalation types the model can predict (new_subcategory is probably easier than severity_increase).

In [ ]:
from src.analysis.plotting import _s4_escalation_type
_s4_escalation_type(results,
                    show=True, save=False, fig_dir=FIGURES_DIR)

**Seed stability** (if multi-seed results exist):
- Probe AUROC dot plot: one dot per seed per vector $x$ label pair, with mean and range
- SAE stability histogram: distribution of matched cosine similarities across seeds, with vertical line at 0.8 threshold
- Cluster persistence: bar chart of adjusted Rand index between each seed pair

In [ ]:
if seed_stability:
    from src.analysis.plotting import _s4_seed_stability
    _s4_seed_stability(seed_stability,
                       show=True, save=False, fig_dir=FIGURES_DIR)